# App Directory — Gemini Decision 1 Benchmark

Interactive notebook to test the two-stage bookmark classification pipeline (`gemini-decision-1` fast-path -> `gemini-2.5-flash` / `flash-lite` fallback) on sample & edge-case websites.

### Setup in Google Colab
1. Open the **Secrets (🔑)** panel in the left sidebar.
2. Add `GEMINI_API_KEY` and toggle **Notebook access** on.
3. Run all cells (`Runtime -> Run all`).

In [ ]:
import os
import json
import time
import requests
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 140)

# Load API key from Colab Secrets first, falling back to environment variable
GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

assert GEMINI_API_KEY, "Missing GEMINI_API_KEY! Add it to Colab Secrets or os.environ."

# Tunable pipeline parameters (mirrors src/extension/decisions.ts)
DECISION_MODEL = "gemini-decision-1"
GENERATIVE_MODEL = "gemini-2.5-flash"  # Try gemini-2.5-flash-lite or gemini-3.5-flash-lite too
CONFIDENCE_THRESHOLD = 0.75
NEW_TAG_ROUTE_THRESHOLD = 0.50
MAX_RECOMMENDED_TAGS = 5

print(f"Configured: decision={DECISION_MODEL}, fallback={GENERATIVE_MODEL}, "
      f"tag_threshold={CONFIDENCE_THRESHOLD}, new_tag_threshold={NEW_TAG_ROUTE_THRESHOLD}")

In [ ]:
# Existing taxonomy categories (mirrors distinctions in src/extension/taxonomy.ts)
CATEGORIES = [
    "AI App Builder",
    "AI Website Builder",
    "Coding Agents",
    "AI Code Review",
    "AI Agent Development",
    "Agents",
    "Multi-Agent Systems",
    "Voice AI",
    "AI Voice",
    "Local AI",
    "AI Workflow Automation",
    "CI/CD",
    "Containerization",
    "AI Model Evaluation",
    "AI Models Aggregator",
    "AI Tools Directory",
    "AI News",
]

# Optional: Load your real categories from an App Directory export JSON uploaded to Colab
# with open("app-directory-export.json", "r", encoding="utf-8") as f:
#     exported = json.load(f)
#     CATEGORIES = sorted({c for e in exported.get("entries", []) for c in e.get("categories", [])})

# Mix of clear existing-tag matches, subtle taxonomy distinctions, sparse text, and novel categories
TEST_SITES = [
    {
        "name": "Lovable",
        "expected": "fast-path (AI App Builder)",
        "url": "https://lovable.dev",
        "title": "Lovable - Build full-stack web apps from prompts",
        "description": "Create full-stack web applications with React, Supabase, and deployment from natural language prompts.",
    },
    {
        "name": "CodeRabbit",
        "expected": "fast-path (AI Code Review)",
        "url": "https://coderabbit.ai",
        "title": "CodeRabbit | AI Code Reviews",
        "description": "Automated line-by-line pull request reviews, bug detection, and security checks for GitHub and GitLab.",
    },
    {
        "name": "Ollama",
        "expected": "fast-path (Local AI)",
        "url": "https://ollama.com",
        "title": "Ollama",
        "description": "Run Llama, DeepSeek, Gemma, and other open-weight LLMs locally on your own machine with a CLI and local REST API.",
    },
    {
        "name": "ElevenLabs",
        "expected": "fast-path (AI Voice)",
        "url": "https://elevenlabs.io",
        "title": "ElevenLabs - AI Voice Generator & Text to Speech",
        "description": "Realistic AI voice generation, voice cloning, dubbing, and audio synthesis for creators and developers.",
    },
    {
        "name": "Vapi",
        "expected": "fast-path (Voice AI)",
        "url": "https://vapi.ai",
        "title": "Vapi - Voice AI Agents for Developers",
        "description": "Build, test, and deploy real-time conversational voice AI agents for phone calls and customer support.",
    },
    {
        "name": "n8n",
        "expected": "fast-path (AI Workflow Automation)",
        "url": "https://n8n.io",
        "title": "n8n - AI Workflow Automation Platform",
        "description": "Visual node-based workflow automation connecting APIs, webhooks, and AI agents.",
    },
    {
        "name": "OpenRouter",
        "expected": "fast-path (AI Models Aggregator)",
        "url": "https://openrouter.ai",
        "title": "OpenRouter - Unified Interface for LLMs",
        "description": "Access, compare, and route across hundreds of AI models from OpenAI, Anthropic, Google, and Meta through one API.",
    },
    {
        "name": "Docker",
        "expected": "fast-path (Containerization)",
        "url": "https://www.docker.com",
        "title": "Docker: Accelerated Container Application Development",
        "description": "Package, ship, and run applications in portable containers across development and production.",
    },
    {
        "name": "Sparse Example (CrewAI)",
        "expected": "fast-path or fallback (Multi-Agent Systems)",
        "url": "https://crewai.com",
        "title": "CrewAI",
        "description": "Framework for orchestrating role-playing autonomous AI agents.",
    },
    {
        "name": "Onshape (Novel Category)",
        "expected": "route to Flash (new tag: 3D CAD / Design)",
        "url": "https://www.onshape.com",
        "title": "Onshape | Cloud-Native CAD & PDM",
        "description": "Browser-based parametric 3D CAD software for mechanical engineers and hardware product teams.",
    },
    {
        "name": "BandLab (Novel Category)",
        "expected": "route to Flash (new tag: Music Production / DAW)",
        "url": "https://www.bandlab.com",
        "title": "BandLab - Make Music Online",
        "description": "Free cloud digital audio workstation (DAW) to record, mix, and master tracks in your browser.",
    },
    {
        "name": "YNAB (Novel Category)",
        "expected": "route to Flash (new tag: Personal Finance)",
        "url": "https://www.ynab.com",
        "title": "YNAB - Zero-Based Budgeting App",
        "description": "Personal budgeting software that helps households allocate every dollar, track bank accounts, and pay off debt.",
    },
]

print(f"Loaded {len(CATEGORIES)} categories and {len(TEST_SITES)} test sites.")

In [ ]:
def format_site_input(site):
    return (
        f"URL: {site['url']}\n"
        f"Title: {site['title']}\n"
        f"Meta Description: {site['description']}"
    )


def run_decisions_stage(site, categories, threshold=CONFIDENCE_THRESHOLD):
    questions = {}
    for idx, cat in enumerate(categories):
        questions[f"tag_{idx}"] = {
            "type": "noul",
            "instructions": (
                f'Does this website\'s core product or primary purpose belong in the "{cat}" category? '
                f'Answer yes only if "{cat}" is a substantial, first-class capability of the site rather than a minor feature.'
            ),
        }

    questions["needs_new_tag"] = {
        "type": "noul",
        "instructions": (
            f"Is this website's primary product or category missing from all of the existing categories "
            f"({', '.join(categories)}), such that a brand-new reusable taxonomy tag is genuinely needed?"
        ),
    }

    t0 = time.time()
    resp = requests.post(
        "https://generativelanguage.googleapis.com/v1beta/decisions",
        headers={"Content-Type": "application/json", "x-goog-api-key": GEMINI_API_KEY},
        json={
            "model": DECISION_MODEL,
            "input": format_site_input(site),
            "questions": questions,
        },
        timeout=30,
    )
    resp.raise_for_status()
    latency_ms = round((time.time() - t0) * 1000)
    answers = resp.json().get("answers", {})

    tag_scores = {}
    for idx, cat in enumerate(categories):
        ans = answers.get(f"tag_{idx}", {})
        tag_scores[cat] = float(ans.get("noul", ans.get("value", 0.0)))

    matched_tags = [
        cat for cat, score in sorted(tag_scores.items(), key=lambda x: -x[1])
        if score >= threshold
    ][:MAX_RECOMMENDED_TAGS]

    new_ans = answers.get("needs_new_tag", {})
    needs_new_tag_score = float(new_ans.get("noul", new_ans.get("value", 0.0)))
    should_route = (len(matched_tags) == 0) or (needs_new_tag_score >= NEW_TAG_ROUTE_THRESHOLD)

    return {
        "matched_tags": matched_tags,
        "needs_new_tag_score": needs_new_tag_score,
        "should_route": should_route,
        "tag_scores": tag_scores,
        "latency_ms": latency_ms,
    }


def run_generative_stage(site, categories, hint_tags):
    hint_block = (
        f"\nPre-matched Existing Tags (from fast classifier): {', '.join(hint_tags)}\n"
        f"Keep any of these that fit, and focus on whether a genuinely distinct new tag is needed.\n"
        if hint_tags else ""
    )
    prompt = (
        f"Classify this website into 1-5 tags. Prefer Existing Categories when they fit; "
        f"propose 1 concise new tag in newTags only if no existing category fits.\n\n"
        f"{format_site_input(site)}\n\n"
        f"Existing Categories: [{', '.join(categories)}]\n"
        f"{hint_block}"
        f"Return JSON: {{\"recommendedTags\": [...], \"newTags\": [...], \"reasoning\": \"...\"}}"
    )

    t0 = time.time()
    resp = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{GENERATIVE_MODEL}:generateContent?key={GEMINI_API_KEY}",
        headers={"Content-Type": "application/json"},
        json={
            "contents": [{"role": "user", "parts": [{"text": prompt}]}],
            "generationConfig": {"response_mime_type": "application/json", "temperature": 0.15},
        },
        timeout=30,
    )
    resp.raise_for_status()
    latency_ms = round((time.time() - t0) * 1000)
    raw_text = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
    parsed = json.loads(raw_text)
    return {
        "recommended_tags": parsed.get("recommendedTags", []),
        "new_tags": parsed.get("newTags", []),
        "reasoning": parsed.get("reasoning", ""),
        "latency_ms": latency_ms,
    }

In [ ]:
rows = []
detailed_runs = {}

for site in TEST_SITES:
    d_res = run_decisions_stage(site, CATEGORIES)
    total_ms = d_res["latency_ms"]
    final_tags = d_res["matched_tags"]
    new_tags = []

    if d_res["should_route"]:
        g_res = run_generative_stage(site, CATEGORIES, d_res["matched_tags"])
        total_ms += g_res["latency_ms"]
        final_tags = g_res["recommended_tags"]
        new_tags = g_res["new_tags"]

    top_3 = [
        f"{c} ({s:.2f})"
        for c, s in sorted(d_res["tag_scores"].items(), key=lambda x: -x[1])[:3]
    ]

    detailed_runs[site["name"]] = d_res
    rows.append({
        "site": site["name"],
        "expected": site["expected"],
        "decision_tags": ", ".join(d_res["matched_tags"]) or "-",
        "new_tag_score": round(d_res["needs_new_tag_score"], 2),
        "routed_to_flash": d_res["should_route"],
        "final_tags": ", ".join(final_tags),
        "new_tags": ", ".join(new_tags) or "-",
        "decision_ms": d_res["latency_ms"],
        "total_ms": total_ms,
        "top_3_scores": " | ".join(top_3),
    })

df = pd.DataFrame(rows)
fast_path_pct = round((~df["routed_to_flash"]).mean() * 100, 1)
print(f"Fast-path hit rate: {fast_path_pct}% | Avg Decision latency: {df['decision_ms'].mean():.0f} ms | Avg Total latency: {df['total_ms'].mean():.0f} ms")
df

In [ ]:
def inspect_site(site_name):
    res = detailed_runs[site_name]
    print(f"=== {site_name} (needs_new_tag={res['needs_new_tag_score']:.2f}, routed={res['should_route']}) ===")
    score_df = pd.DataFrame(
        sorted(res["tag_scores"].items(), key=lambda x: -x[1]),
        columns=["category", "score"]
    )
    return score_df

# Change the name to inspect any site's full category score distribution
inspect_site("Onshape (Novel Category)")